# PyRosetta Scoring Functions

Compute physics-based scoring metrics for protein structures using PyRosetta:
- **SAP** (Spatial Aggregation Propensity) — surface hydrophobicity / aggregation risk
- **SASA** (Solvent Accessible Surface Area) — solvent exposure per residue and total
- **Energy** — Rosetta energy scoring with optional FastRelax

**License:** PyRosetta requires acceptance of the [Rosetta Software License](https://www.rosettacommons.org/software/license-and-download).
Free for academic/non-commercial use.

In [1]:
from proto_tools.tools.structure_scoring.pyrosetta import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaSAPInput,
    PyRosettaSASAConfig,
    PyRosettaSASAInput,
    run_pyrosetta_energy,
    run_pyrosetta_sap,
    run_pyrosetta_sasa,
)

pdb_path = "example.pdb"

## SAP Scoring

SAP quantifies surface hydrophobicity / aggregation propensity. Higher = more aggregation-prone.

In [2]:
sap_result = run_pyrosetta_sap(
    PyRosettaSAPInput(inputs=[pdb_path])
)
print(f"SAP score: {sap_result.results[0].sap_score:.2f}")

SAP score: 50.62


## SASA Computation

Compute total and per-residue solvent accessible surface area.

In [3]:
sasa_result = run_pyrosetta_sasa(
    PyRosettaSASAInput(inputs=[pdb_path]),
    PyRosettaSASAConfig(probe_radius=1.4),
)
print(f"Total SASA: {sasa_result.results[0].total_sasa:.1f} A^2")
print(f"Residues: {len(sasa_result.results[0].per_residue)}")

# Top 5 most exposed residues
sorted_res = sorted(sasa_result.results[0].per_residue, key=lambda r: r.sasa, reverse=True)
for res in sorted_res[:5]:
    print(f"  {res.chain_id}:{res.residue_name}{res.residue_index} — {res.sasa:.1f} A^2")

Total SASA: 4978.9 A^2
Residues: 65
  A:LYS46 — 235.9 A^2
  A:ARG21 — 214.8 A^2
  A:MET1 — 195.9 A^2
  A:PRO65 — 186.1 A^2
  A:LYS35 — 179.9 A^2


## Energy Scoring

Compute Rosetta energy with optional FastRelax to resolve clashes.

In [4]:
energy_result = run_pyrosetta_energy(
    PyRosettaEnergyInput(inputs=[pdb_path]),
    PyRosettaEnergyConfig(scorefxn="ref2015", relax=True, relax_cycles=1),
)
result = energy_result.results[0]
print(f"Total energy: {result.total_energy:.1f} REU")
print(f"Relaxed: {result.relaxed}")
print(f"\nEnergy terms:")
for term, value in sorted(result.energy_terms.items(), key=lambda x: x[1]):
    print(f"  {term}: {value:.2f}")

Total energy: -189.3 REU
Relaxed: True

Energy terms:
  fa_atr: -337.53
  fa_elec: -116.02
  p_aa_pp: -30.60
  hbond_sr_bb: -22.27
  hbond_lr_bb: -15.69
  hbond_sc: -9.88
  lk_ball_wtd: -7.06
  rama_prepro: -6.41
  hbond_bb_sc: -5.84
  dslf_fa13: 0.00
  yhh_planarity: 0.09
  pro_close: 0.91
  ref: 4.91
  omega: 5.56
  fa_intra_sol_xover4: 11.45
  fa_rep: 74.06
  fa_dun: 117.47
  fa_intra_rep: 145.82
  fa_sol: 202.73


## Export Results

In [5]:
# Export SASA results to CSV
sasa_result.export(name="sasa_results", export_path="./example_output", file_format="csv")

# Export energy results to JSON
energy_result.export(name="energy_results", export_path="./example_output", file_format="json")